In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from pandas.api.types import *

In [ ]:
def load_and_validate_seoul_bikes(csv_path: str | Path) -> pd.DataFrame:
    
    raw_df = pd.read_csv(csv_path)

    def to_timestamp(date: pd.Series, hour: pd.Series) -> pd.Series:
        hour = pd.to_numeric(hour, errors="coerce")
        if (hour.isna() | (hour % 1 != 0) | (hour < 0) | (hour > 23)).any():
            raise ValueError("Invalid Hour: Hour must be an integer between 0 and 23")
        return pd.to_datetime(date, format="%d/%m/%Y") + pd.to_timedelta(hour, unit="h")

    # Create a new DataFrame with a timestamp column and drop the original Date and Hour columns
    with_timestamp = raw_df.assign(
        timestamp = to_timestamp(raw_df["Date"], raw_df["Hour"])
    ).drop(columns=["Date", "Hour"]).set_index("timestamp")

    with_timestamp = with_timestamp.rename(columns={
        "Rented Bike Count": "rented_bikes",
        "Temperature(C)": "temperature_c",
        "Humidity(%)": "humidity_pct",
        "Wind speed (m/s)": "wind_speed_m_s",
        "Visibility (10m)": "visibility_10m",
        "Dew point temperature(C)": "dew_point_c",
        "Solar Radiation (MJ/m2)": "solar_radiation_mj_m2",
        "Rainfall(mm)": "rainfall_mm",
        "Snowfall (cm)": "snowfall_cm",
        "Seasons": "season",
        "Holiday": "holiday",
        "Functioning Day": "functioning_day"
    })

    with_timestamp = with_timestamp.sort_index(ascending=True).reset_index()

    numeric_columns = [
        "rented_bikes",
        "temperature_c",
        "humidity_pct",
        "wind_speed_m_s",
        "visibility_10m",
        "dew_point_c",
        "solar_radiation_mj_m2",
        "rainfall_mm",
        "snowfall_cm"
    ]

    # Validate numeric columns
    for col in numeric_columns:
        with_timestamp[col] = pd.to_numeric(with_timestamp[col], errors="coerce")
        if with_timestamp[col].isna().any(): 
            raise ValueError(f"Invalid values found in column '{col}'")
        if col == "rented_bikes": 
            if ((with_timestamp[col] % 1 != 0) | (with_timestamp[col] < 0)).any():
                raise ValueError(f"{col} must only have integer records.")
        else: 
            if not np.isfinite(with_timestamp[col]).all():
                raise ValueError(f"{col}'s value is infinite.")

    #reject duplicate timestamps
    if with_timestamp["timestamp"].duplicated().any():
        raise ValueError("Duplicate timestamps found.")

    # reject > 1hr gaps in the data, report the first gap found
    time_diff = with_timestamp["timestamp"].diff().dt.total_seconds().div(3600)
    if (time_diff > 1).any():
        first_gap_index = time_diff[time_diff > 1].index[0]
        raise ValueError(f"Data has gaps greater than 1 hour starting at index {first_gap_index}.")

    # reject unknown holiday /functioning day labels
    valid_holiday_labels = {"Holiday", "No Holiday"}
    valid_functioning_day_labels = {"Yes", "No"}
    valid_season_labels = {"Spring", "Summer", "Autumn", "Winter"}
    if not with_timestamp["holiday"].isin(valid_holiday_labels).all():
        raise ValueError("Invalid values found in 'holiday' column.")
    if not with_timestamp["functioning_day"].isin(valid_functioning_day_labels).all():
        raise ValueError("Invalid values found in 'functioning_day' column.")
    if not with_timestamp["season"].isin(valid_season_labels).all():
        raise ValueError("Invalid values found in 'season' column.")
    
    return with_timestamp

In [31]:
load_and_validate_seoul_bikes("../data/SeoulBikeData.csv")

,timestamp,rented_bikes,temperature_c,humidity_pct,wind_speed_m_s,visibility_10m,dew_point_c,solar_radiation_mj_m2,rainfall_mm,snowfall_cm,season,holiday,functioning_day
0,2017-12-01 00:00:00,254,-5.2,37,2.2,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
1,2017-12-01 01:00:00,204,-5.5,38,0.8,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
2,2017-12-01 02:00:00,173,-6.0,39,1.0,2000,-17.7,0.0,0.0,0.0,Winter,No Holiday,Yes
3,2017-12-01 03:00:00,107,-6.2,40,0.9,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
4,2017-12-01 04:00:00,78,-6.0,36,2.3,2000,-18.6,0.0,0.0,0.0,Winter,No Holiday,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8755,2018-11-30 19:00:00,1003,4.2,34,2.6,1894,-10.3,0.0,0.0,0.0,Autumn,No Holiday,Yes
8756,2018-11-30 20:00:00,764,3.4,37,2.3,2000,-9.9,0.0,0.0,0.0,Autumn,No Holiday,Yes
8757,2018-11-30 21:00:00,694,2.6,39,0.3,1968,-9.9,0.0,0.0,0.0,Autumn,No Holiday,Yes
8758,2018-11-30 22:00:00,712,2.1,41,1.0,1859,-9.8,0.0,0.0,0.0,Autumn,No Holiday,Yes
